In [ ]:
#@title &nbsp;
# The banner is a cell output rather than a text cell, because Colab
# limits text cells to a reading width and outputs use the full window.
from IPython.display import HTML, display
display(HTML('<style>\n.adcp-hdr{box-sizing:border-box;width:100%;display:flex;align-items:center;\n  gap:clamp(14px,2.2vw,34px);padding:clamp(14px,2vw,28px) clamp(16px,2.6vw,40px);\n  border-radius:12px;background:linear-gradient(90deg,#15424e,#0b2e37);overflow:hidden;\n  font-family:system-ui,-apple-system,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif}\n.adcp-mark{flex:0 0 auto;width:clamp(58px,6.4vw,112px);height:clamp(58px,6.4vw,112px);\n  border-radius:22%;background:#134c55;display:flex;align-items:center;justify-content:center}\n.adcp-mark svg{width:80%;height:80%}\n.adcp-txt{flex:1 1 auto;min-width:0}\n.adcp-title{color:#f7f9f9;font-weight:700;font-size:clamp(24px,3.2vw,50px);line-height:1.08}\n.adcp-tag{color:#9fc4c4;font-size:clamp(12.5px,1.25vw,19px);margin-top:.45em}\n.adcp-rule{width:clamp(56px,6vw,110px);height:4px;border-radius:2px;background:#d67a3a;\n  margin:clamp(9px,1vw,15px) 0 clamp(7px,.8vw,12px)}\n.adcp-sub{color:#9fc4c4;font-size:clamp(12px,1.05vw,16px)}\n.adcp-motif{flex:0 1 auto;width:clamp(120px,22vw,420px);align-self:flex-end}\n@media (max-width:760px){.adcp-motif{display:none}}\n.adcp-note{font:13.5px/1.6 system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;margin:12px 2px 2px}\n.adcp-note a{color:#1a8a8a}\n.adcp-small{font-size:12px;opacity:.75}\n</style>\n<div class="adcp-hdr">\n  <div class="adcp-mark"><svg viewBox="0 0 62 62" xmlns="http://www.w3.org/2000/svg" aria-hidden="true"><path d="M46 14 A20 20 0 1 0 46 48" fill="none" stroke="#22a0a0" stroke-width="7" stroke-linecap="round"/><path d="M22 40 L30 35 L38 30 L46 25" fill="none" stroke="#f7f9f9" stroke-width="2.6" stroke-linecap="round"/><circle cx="22" cy="40" r="4.2" fill="#d67a3a"/><circle cx="30" cy="35" r="4.2" fill="#f7f9f9"/><circle cx="38" cy="30" r="4.2" fill="#d67a3a"/><circle cx="46" cy="25" r="4.2" fill="#f7f9f9"/></svg></div>\n  <div class="adcp-txt">\n    <div class="adcp-title">GUI to ADCP Docking</div>\n    <div class="adcp-tag">a product of the Scripps Research Institute and the GUI powered by HOPE Lab</div>\n    <div class="adcp-rule"></div>\n    <div class="adcp-sub">Peptide docking, from receptor to ranked table.</div>\n  </div>\n  <svg class="adcp-motif" viewBox="0 0 420 72" preserveAspectRatio="xMaxYMax meet" aria-hidden="true"><polyline points="12,58 78,18 144,58 210,18 276,58 342,18 408,58" fill="none" stroke="#155e65" stroke-width="5" stroke-linejoin="round"/><circle cx="12" cy="58" r="9" fill="#7d5f41"/><circle cx="78" cy="18" r="9" fill="#16686d"/><circle cx="144" cy="58" r="9" fill="#7d5f41"/><circle cx="210" cy="18" r="9" fill="#16686d"/><circle cx="276" cy="58" r="9" fill="#7d5f41"/><circle cx="342" cy="18" r="9" fill="#16686d"/><circle cx="408" cy="58" r="9" fill="#7d5f41"/></svg>\n</div>\n<div class="adcp-note">A graphical interface to <a href="https://ccsb.scripps.edu/adcp/" target="_blank">ADCP</a>, a product of the Scripps Research Institute. Powered by HOPE Lab. Not affiliated with or endorsed by Scripps.<br>\nPress &#9654; on step 1, then on step 2. The list of runs includes a finished example, MDM2 with three peptides, to open and explore. The docking runs on this Colab machine, which suits a trial run; for larger searches, install it on a <a href="https://github.com/aadhilhaq/ADCP_docking#slurm-cluster" target="_blank">Slurm cluster</a>.<br>\n<span class="adcp-small">Step 1 downloads ADFRsuite, which contains ADCP, from the Scripps Research Institute, and optionally AmberTools from conda-forge. Parts of ADFRsuite are free for academic use only.</span></div>\n'))


In [ ]:
#@title **1 · Install** &nbsp; <font color="#5f7070">about 5 minutes, once per session; a few more with MM-GBSA</font>
add_MM_GBSA = True #@param {type:"boolean"}
add_ADCP_1_1 = False #@param {type:"boolean"}
#@markdown Tick **add_MM_GBSA** to install AmberTools for MM-GBSA rescoring, and **add_ADCP_1_1** to add ADCP 1.1 (non-standard and D amino acids, OpenMM minimization; about ten minutes more).
import subprocess, sys, time
from IPython.display import HTML, display

# One status line, updated in place, so the cell ends with a single line.
_status = display(HTML(""), display_id=True)

def _show(text, colour="#5f7070"):
    _status.update(HTML('<div style="font:14px system-ui;color:%s;margin:2px 0">'
                        '%s</div>' % (colour, text)))

def _install():
    steps = [
        ("Installing the interface",
         [sys.executable, "-m", "pip", "-q", "install", "rdkit", "pdb2pqr",
          "openmm", "pdbfixer",
          "adcp-dock @ git+https://github.com/aadhilhaq/ADCP_docking"]),
        # ADFRsuite comes from the Scripps Research Institute, as they ask,
        # and is checked against a pinned SHA-256 before it is unpacked.
        ("Downloading ADFRsuite from the Scripps Research Institute",
         [sys.executable, "-m", "adcp_dock.adfr",
          "--dest", "/content/tools/ADFRsuite"]),
    ]
    if add_MM_GBSA:
        # AmberTools is on conda-forge, not pip. micromamba installs it into
        # its own folder, so Colab's Python and the notebook are untouched.
        steps.append((
            "Installing AmberTools for MM-GBSA from conda-forge",
            ["bash", "-c",
             "set -e; mkdir -p /content/micromamba; cd /content/micromamba; "
             "[ -x bin/micromamba ] || curl -Ls "
             "https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba; "
             "[ -x /content/amber/bin/tleap ] || ./bin/micromamba create -y -q "
             "-r /content/micromamba -p /content/amber -c conda-forge ambertools"]))
    if add_ADCP_1_1:
        # Scripps' own recipe, as a separate Python 3.7 environment; the
        # interface finds it at /content/adcpsuite.
        steps.append((
            "Installing ADCP 1.1 from the Scripps Research Institute",
            [sys.executable, "-m", "adcp_dock.adcp11_install",
             "--dest", "/content/adcpsuite"]))
    steps = [("%s (%d of %d)" % (label, i + 1, len(steps)), cmd)
             for i, (label, cmd) in enumerate(steps)]
    for label, cmd in steps:
        _show(label + " …")
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode:
            _show("✗ %s failed. Its output is below." % label.split(" (")[0],
                  "#b3412e")
            print((p.stdout + p.stderr)[-4000:])
            return False
    return True

_t0 = time.time()
if _install():
    _m, _s = divmod(int(time.time() - _t0), 60)
    _took = ("%d min %02d s" % (_m, _s)) if _m else ("%d s" % _s)
    _extra = [x for x, on in (("MM-GBSA", add_MM_GBSA), ("ADCP 1.1", add_ADCP_1_1)) if on]
    _show("✓ Installed in %s%s" % (_took, (", with " + " and ".join(_extra)) if _extra else ""),
          "#1a8a8a")


In [ ]:
#@title **2 · Open GUI to ADCP Docking**
import importlib, importlib.util, os, re, subprocess, sys, time
from IPython.display import HTML, Javascript, display
from google.colab import output

PORT = 8765
RUNS = "/content/adcp_runs"
ADFR = "/content/tools/ADFRsuite"
TOKEN = None

def _note(text, colour="#b3412e"):
    display(HTML('<div style="font:14px system-ui;color:%s;margin:2px 0">%s</div>'
                 % (colour, text)))

importlib.invalidate_caches()
if importlib.util.find_spec("adcp_dock") is None or not os.path.isdir(ADFR):
    _note("Run step 1 first.")
else:
    os.makedirs(RUNS, exist_ok=True)
    # A finished example run to open before docking anything: MDM2 with
    # three peptides. Left as it is when it is already there.
    _ex = subprocess.run([sys.executable, "-m", "adcp_dock.cli", "example",
                          "--runs", RUNS], capture_output=True, text=True)
    if _ex.returncode:
        _note("The example run could not be added: %s"
              % (_ex.stdout + _ex.stderr).strip()[-300:])
    # Running this cell again restarts the server instead of adding a second.
    if globals().get("_SERVER") and _SERVER.poll() is None:
        _SERVER.terminate()
        _SERVER.wait(10)
    # ADCP_EXECUTOR=local runs the work on this machine; Colab has no SLURM.
    env = dict(os.environ, ADCP_EXECUTOR="local", ADFRSUITE_HOME=ADFR,
               PATH=ADFR + "/bin:" + os.environ["PATH"])
    # AmberTools from step 1, for MM-GBSA. Appended to PATH, so its own
    # python and tools never shadow Colab's.
    if os.path.isfile("/content/amber/bin/tleap"):
        env["AMBERHOME"] = "/content/amber"
        env["PATH"] = env["PATH"] + ":/content/amber/bin"
    _SERVER = subprocess.Popen(
        [sys.executable, "-u", "-m", "adcp_dock.cli", "serve",
         "--host", "127.0.0.1", "--port", str(PORT), "--runs", RUNS],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    seen, deadline = [], time.time() + 60
    while time.time() < deadline and TOKEN is None:
        line = _SERVER.stdout.readline()
        if not line:
            if _SERVER.poll() is not None:
                break
            continue
        seen.append(line)
        m = re.search(r"http://127\.0\.0\.1:\d+/\?t=([A-Za-z0-9_\-]+)", line)
        if m:
            TOKEN = m.group(1)

    if TOKEN is None:
        _note("The interface did not start. Its output is below.")
        print("".join(seen[-30:]))
    else:
        try:
            output.no_vertical_scroll()
        except Exception:
            pass
        output.serve_kernel_port_as_window(
            PORT, path="/?t=" + TOKEN, anchor_text="Open in a separate tab ↗")
        # embed=1 leaves the banner above to this notebook and lets the form
        # use the full width of the frame.
        output.serve_kernel_port_as_iframe(
            PORT, path="/?t=%s&embed=1" % TOKEN, width="100%", height="800")
        # Size the frame to the visible window, so the interface scrolls
        # inside it like an application window.
        display(Javascript("""
(function fit(tries) {
  var f = document.querySelector('iframe');
  if (!f) { if (tries < 100) setTimeout(function () { fit(tries + 1); }, 150); return; }
  f.style.width = '100%';
  f.style.display = 'block';
  f.style.border = '1px solid #d5e0e0';
  f.style.borderRadius = '10px';
  var size = function () {
    f.style.height = Math.max(620, (window.outerHeight || screen.availHeight) - 230) + 'px';
  };
  size();
  window.addEventListener('resize', size);
})(0);
"""))


In [ ]:
#@title **3 · Download a run** &nbsp; <font color="#5f7070">optional: the interface has its own download buttons</font>
RUN = "" #@param {type:"string"}
#@markdown Leave empty for the latest run. The zip includes HOW_TO_CITE.txt, CITATION.bib and CITATION.ris.
import os, shutil
from IPython.display import HTML, display
from google.colab import files

ROOT = "/content/adcp_runs"
runs = sorted((os.path.join(ROOT, d) for d in
               (os.listdir(ROOT) if os.path.isdir(ROOT) else [])
               if os.path.isfile(os.path.join(ROOT, d, "job.json"))),
              key=os.path.getmtime)
if not runs:
    display(HTML('<div style="font:14px system-ui;color:#b3412e">'
                 'No runs yet. Start one in step 2.</div>'))
else:
    pick = os.path.join(ROOT, RUN) if RUN else runs[-1]
    z = shutil.make_archive("/content/" + os.path.basename(pick), "zip", pick)
    display(HTML('<div style="font:14px system-ui;color:#1a8a8a">✓ %s (%.1f MB)</div>'
                 % (os.path.basename(z), os.path.getsize(z) / 1e6)))
    files.download(z)
